In [3]:
import numpy as np


# 이 클래스는 bandit 설정을 담고 있음.
# probs와 seed를 기본입력으로 넣음.
# 즉, bandit의 확률이 설정된, 그러한 bandit 기계라고 생각하면됨.
class BernoulliBandit:

    def __init__(self, probs: list[float], seed: int = 0):
        self._probs = np.array(probs, dtype=float, copy=True)

        if self._probs.ndim != 1 or self._probs.size == 0:
            raise ValueError("probs should be occupied 1-d array")

        # 모든 배열의 원소가 전부 True이다 + finite을 true로
        if not np.all(np.isfinite(self._probs)):
            raise ValueError("all elements should be finite")

        # 0보다 작거나, 1보다 큰 원소를 false로 변환.
        if np.any( (self._probs <0) | (self._probs > 1) ):
            raise ValueError("all element should be larger than 0, less than 1")

        # arm의 개수 = prob 배열의 개수
        self.n_arms = len(self._probs)

        # 랜덤생성기
        self.rng = np.random.default_rng(seed)


    def reward(self, action: int) -> float:
        """
        Input : bandit number  (  0 <= action < n_arms )

        Output : the reward from input action,  0.0  or 1.0
        """

        if ( (action<0) or (action>=self.n_arms) ):
            raise ValueError(f'invalid arm number; 0 <= arm number < {self.n_arms}')

        random_number = self.rng.random()

        success_prob = self._probs[action]

        reward = float(self.rng.binomial(1, success_prob))

        return reward

    def true_means(self) -> np.ndarray:
        return self._probs.copy()


class ThompsonSampling:
    def __init__(self, n_arms: int, seed: int = 1):
        if n_arms < 1:
            raise ValueError("n_arms should be larger than 1")

        self.n_arms = n_arms
        self.rng = np.random.default_rng(seed)

        # 각 arm의 초기 prior: Beta(1, 1) = Uniform dist.
        # 따라서, 모든 arm에 대한 alpha,beta를 1로 초기화.
        # 이 값들은 update 함수를 통해서 계속 업데이트 된다.
        self.alpha = np.ones(n_arms, dtype=float)
        self.beta = np.ones(n_arms, dtype=float)

    def select_action(self) -> int:
        """
        Using current posteriors of arms, sample each parameter to return 
        argmax of parameter vector.

        no input

        output : select arm number 
        """

        sample_prob = self.rng.beta(self.alpha, self.beta)

        action = int(np.argmax(sample_prob))

        return action


    def update(self, action: int , reward: float) -> None:
        """
        input : selected arm number, and the reward of the given action.

        output : None. just update alpha and beta parameter
        """

        self.alpha[action] = self.alpha[action] + reward
        self.beta[action] =  self.beta[action] + (1-reward)


    

In [4]:
def run_bandit(env : BernoulliBandit, agent: ThompsonSampling, steps: int) -> dict[str, np.ndarray]:
    """
    connects env and agents and let them interact 'steps' time.

    input:  
        env : BernoulliBandit class
        agent : ThompsonSampling agent
        steps : how many times do they interact

    output:
        {
            "actions" : history of selected action, 
            "rewards" :  history of rewards, 
            "instant_regret" : regret at t, 
            "cumulative_regret" : summation of regret
        }
    """
    if steps < 1 :
        raise ValueError("steps should be positive")

    actions = np.empty(steps, dtype=int)
    rewards = np.empty(steps, dtype=float)
    instant_regret = np.empty(steps, dtype=float)

    # this values are "true parameter and optimal action"
    # should not be known to agent.
    means = env.true_means()
    best_mean = float(means.max())


    for t in range(steps):
        """
        1. action selection from agent
        2. what is reward of the action?
        3. update prior from the data to get posterior
        4. go to 1
        """

        action = agent.select_action()
        actions[t] = action

        reward = env.reward(action)
        rewards[t] = reward

        instant_regret[t] = best_mean - means[action]

        agent.update(action, reward)


    return{
        "actions" : actions,
        "rewards" : rewards,
        "instant_regret" : instant_regret,
        "cumulative_regret" : np.cumsum(instant_regret)
    }





In [5]:
env = BernoulliBandit([0.2, 0.5, 0.8], seed = 0)
agent = ThompsonSampling(3, seed=1)

result = run_bandit(env, agent, steps=1000)

print("mean reward:", result["rewards"].mean())

print("cumulative pseudo-regret:", result["cumulative_regret"][-1])



mean reward: 0.795
cumulative pseudo-regret: 6.300000000000001
